# Bout-to-bout transition matrices

Companion to `explore_calls_xplatform.ipynb` (which builds **call → call** transition
matrices). Here the main unit is a **bout**, focused on the **2–300 s** inter-bout
timescale.

**Pipeline**
1. Divide calls into bouts with the repo's single source of truth,
   `vocalization_analysis.bouts.detect_bouts` (per call type).
2. Keep only **true bouts** (`in_bout`, i.e. ≥ `min_bout_size` calls).
3. Collapse each bout to one event (type, start, stop + audio coordinates).
4. Order bouts in time within each `(date_folder, exp, assigned_location)` group and
   count **bout → bout transitions** whose inter-bout silent gap is in **(2 s, 300 s]**.

Call types with a bout threshold: **warble, high-freq, alarm, stacks** → 4×4 matrix.
(`alarm` and `stacks` use a `< 2 s` gap; `warble`/`high-freq` use `0.05–0.20 s`.)

**Sections:** pooled matrix · arena vs underground · **tau-resolved transition curves
(call-level and bout-level)** · QC example spectrograms · gap distribution.

In [ ]:
import platform
import sys
import functools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Platform-aware paths (same convention as explore_calls_xplatform.ipynb) ---
HOST = platform.system()  # "Darwin" on Mac, "Linux" on the cluster
if HOST == "Darwin":
    DROPBOX     = Path("/Users/gilyginosar/Dropbox (Personal)/Vocalizations_project")
    PARQUET_DIR = DROPBOX / "Data" / "parquet_cache"
    REPO_ROOT   = Path("/Users/gilyginosar/repos/gerbil_vocalization_analysis")
    BASE_AUDIO  = None  # raw WAVs aren't on the Mac -> the QC spectrogram cells are skipped
elif HOST == "Linux":
    BASE_AUDIO  = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio")
    PARQUET_DIR = BASE_AUDIO / "all_calls" / "parquet_cache"
    REPO_ROOT   = Path("/mnt/home/gginosar/repos/gerbil_vocalization_analysis")
else:
    raise RuntimeError(f"Unsupported platform: {HOST}")

# Shared bout-detection helpers (single source of truth for thresholds).
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from vocalization_analysis.bouts import BOUT_THRESHOLDS, DEFAULT_GROUP_COLS, detect_bouts

print(f"HOST={HOST}")
print(f"PARQUET_DIR={PARQUET_DIR}")

In [ ]:
# --- Analysis settings -------------------------------------------------------
DATES = ["2025_03", "2025_07", "2025_10", "2026_02"]   # same pool as the call version

# Inter-bout timescale of interest for the fixed-band matrix.
GAP_LO_S = 2      # exclusive lower bound (seconds)
GAP_HI_S = 300    # inclusive upper bound (seconds)

# Call types with a bout definition in BOUT_THRESHOLDS; rows/columns of every matrix.
BOUT_CALL_TYPES = list(BOUT_THRESHOLDS)          # ["warble", "high-freq", "alarm", "stacks"]

# A bout never spans across these columns (repo default).
GROUP_COLS = list(DEFAULT_GROUP_COLS)            # ["date_folder", "exp", "assigned_location"]

# 2-level location grouping for the arena-vs-underground split.
LOC_GROUPS = {"arena_1": "arena", "arena_2": "arena", "underground": "underground"}

# One colour per call type, reused by every line plot.
TYPE_COLORS = {"warble": "#2A9D8F", "high-freq": "#457B9D",
               "alarm": "#E63946", "stacks": "#E9C46A"}

## How a bout is defined (single source of truth)

Taken verbatim from `vocalization_analysis/bouts.py` — thresholds are **not** redefined
here, so retuning there flows through automatically:

| call type | min silent gap | max silent gap | min calls |
|-----------|----------------|----------------|-----------|
| warble    | 0.05 s         | 0.20 s         | 5         |
| high-freq | 0.05 s         | 0.20 s         | 5         |
| alarm     | – (no lower)   | 2.0 s          | 5         |
| stacks    | – (no lower)   | 2.0 s          | 5         |

A **bout** is a run of consecutive **same-type** calls within one
`(date_folder, exp, assigned_location)` group, where each silent gap
`icg = this.start − prev.stop` lies inside `[min_gap, max_gap]`. A gap outside that
window starts a new bout. We keep only `in_bout` runs (≥ `min_bout_size` calls).

In [ ]:
# The exact thresholds in force for this run (for the record).
pd.DataFrame(BOUT_THRESHOLDS).T[["min_icg_s", "max_icg_s", "min_bout_size"]]

## Load and pool the calls across dates

We keep the audio-locating columns (`file_num`, `channel`, `start_time_file_sec`,
`stop_time_file_sec`) so the QC section can pull raw spectrograms later.

In [ ]:
NEEDED = ["event_type", *GROUP_COLS, "start_time_real", "stop_time_real",
          "file_num", "channel", "start_time_file_sec", "stop_time_file_sec"]

calls = pd.concat(
    [pd.read_parquet(PARQUET_DIR / f"all_calls_{d}.parquet", columns=NEEDED)
     for d in DATES],
    ignore_index=True,
)
for col in ("start_time_real", "stop_time_real"):
    calls[col] = pd.to_datetime(calls[col], errors="coerce")
calls = calls.dropna(subset=["event_type", "start_time_real", "stop_time_real"])

print(f"{len(calls):,} calls pooled across {DATES}")
calls["event_type"].value_counts()

## Step 1 — divide calls into bouts, keep only true bouts

For each call type we run `detect_bouts` (it pulls its gap thresholds + `min_bout_size`
from `BOUT_THRESHOLDS`) and keep the `in_bout` runs. Then we collapse each bout to one
row: its **type**, **span** (`start_time`/`stop_time`), call count, and the audio
coordinates of its **first** and **last** call (so QC can render its start and end).

In [ ]:
tagged = pd.concat(
    [detect_bouts(calls[calls["event_type"] == ct], ct, group_cols=GROUP_COLS)
       .query("bout_kind == 'in_bout'")
     for ct in BOUT_CALL_TYPES
     if (calls["event_type"] == ct).any()],
    ignore_index=True,
)

# Collapse to one row per bout. bout_id is unique only within a call type, so the bout
# key includes event_type. First/last calls (by start) give the span + audio coords.
BOUT_KEY = ["event_type", *GROUP_COLS, "bout_id"]
tagged = tagged.sort_values([*BOUT_KEY, "start_time_real"])
first  = tagged.drop_duplicates(BOUT_KEY, keep="first")
last   = tagged.drop_duplicates(BOUT_KEY, keep="last")

bouts = (
    first[[*BOUT_KEY, "start_time_real", "file_num", "channel", "start_time_file_sec"]]
    .rename(columns={"start_time_real": "start_time", "file_num": "first_file",
                     "channel": "first_ch", "start_time_file_sec": "first_fsec"})
    .merge(
        last[[*BOUT_KEY, "stop_time_real", "file_num", "channel", "stop_time_file_sec"]]
        .rename(columns={"stop_time_real": "stop_time", "file_num": "last_file",
                         "channel": "last_ch", "stop_time_file_sec": "last_fsec"}),
        on=BOUT_KEY,
    )
    .merge(tagged.groupby(BOUT_KEY).size().rename("n_calls").reset_index(), on=BOUT_KEY)
)

print(f"{len(bouts):,} true bouts (in_bout, >= min_bout_size calls)")
bouts.groupby("event_type").size()

## Step 2 — count bout → bout transitions in the 2–300 s band

Within each group, bouts are ordered by start time. For each adjacent pair the inter-bout
**silent gap** is `next.start − curr.stop`; pairs in **(2 s, 300 s]** increment
`counts[current_type, next_type]`, pooled across groups, then row-normalised to
`P(next bout type | current bout type)`.

In [ ]:
def compute_bout_transitions(bouts, type_order, gap_lo_s, gap_hi_s, group_cols):
    """Pooled bout -> bout counts for transitions with gap in (gap_lo_s, gap_hi_s]."""
    counts = pd.DataFrame(0, index=type_order, columns=type_order, dtype=int)
    for _, g in bouts.groupby(list(group_cols)):
        g = g.sort_values("start_time")
        types  = g["event_type"].to_numpy()
        starts = g["start_time"].to_numpy()
        stops  = g["stop_time"].to_numpy()
        gaps   = (starts[1:] - stops[:-1]) / np.timedelta64(1, "s")
        for i in np.flatnonzero((gaps > gap_lo_s) & (gaps <= gap_hi_s)):
            counts.loc[types[i], types[i + 1]] += 1
    return counts


counts = compute_bout_transitions(bouts, BOUT_CALL_TYPES, GAP_LO_S, GAP_HI_S, GROUP_COLS)
probs  = counts.div(counts.sum(axis=1), axis=0).fillna(0)   # P(next bout | current bout)

print(f"n_transitions ({GAP_LO_S}-{GAP_HI_S} s) = {int(counts.values.sum()):,}")
counts

## Bout → bout transition matrix (2–300 s, all locations pooled)

In [ ]:
def plot_transition_matrix(probs, counts, type_order, title, ax=None):
    """Heatmap of P(next | current) with per-cell probabilities annotated."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    m = probs.values
    im = ax.imshow(m, cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(range(len(type_order))); ax.set_xticklabels(type_order, rotation=40, ha="right")
    ax.set_yticks(range(len(type_order))); ax.set_yticklabels(type_order)
    for i in range(len(type_order)):
        for j in range(len(type_order)):
            ax.text(j, i, f"{m[i, j]:.2f}", ha="center", va="center",
                    color="white" if m[i, j] > 0.55 else "black", fontsize=10)
    ax.set_title(f"{title}\n(n={int(counts.values.sum()):,})", fontsize=11)
    ax.set_ylabel("current bout"); ax.set_xlabel("next bout")
    return im


fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = plot_transition_matrix(probs, counts, BOUT_CALL_TYPES,
                            f"Bout -> bout, {GAP_LO_S}-{GAP_HI_S} s", ax=ax)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("P(next bout | current bout)")
fig.tight_layout()
plt.show()

## Arena vs underground

Same band, computed separately per location group (arena = arena_1 + arena_2). The
matrix above pools locations; this splits them.

In [ ]:
import matplotlib as mpl

def plot_matrix_nan(ax, M, order, title, vmax, cmap_name, *, is_count=False,
                    zero_as_nan=True, fmt="{:.2f}"):
    """Heatmap of M (reindexed to `order`). Blank+grey cells with NaN label where
    the value is NaN, and (if zero_as_nan) where it is 0."""
    A    = M.reindex(index=order, columns=order).to_numpy(dtype=float)
    disp = A.copy()
    if zero_as_nan:
        disp[disp == 0] = np.nan
    cmap = plt.get_cmap(cmap_name).copy(); cmap.set_bad("#f2f2f2")
    im = ax.imshow(disp, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=40, ha="right")
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
    for i in range(len(order)):
        for j in range(len(order)):
            v = disp[i, j]
            if np.isnan(v):
                ax.text(j, i, "NaN", ha="center", va="center", color="#aaa", fontsize=8)
            else:
                lab = f"{int(A[i, j]):,}" if is_count else fmt.format(A[i, j])
                ax.text(j, i, lab, ha="center", va="center",
                        color="white" if v > 0.6 * vmax else "black", fontsize=9)
    ax.set_title(title, fontsize=10); ax.set_xlabel("next bout"); ax.set_ylabel("current bout")
    return im


bouts_loc = bouts.assign(loc2=bouts["assigned_location"].map(LOC_GROUPS))
loc_counts = {loc: compute_bout_transitions(bouts_loc[bouts_loc["loc2"] == loc],
                                            BOUT_CALL_TYPES, GAP_LO_S, GAP_HI_S, GROUP_COLS)
              for loc in ["arena", "underground"]}
count_vmax = max(c.to_numpy().max() for c in loc_counts.values())

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for k, loc in enumerate(["arena", "underground"]):
    c = loc_counts[loc]
    r = c.div(c.sum(axis=1), axis=0)                         # row-normalized P(next|current)
    # Top: raw counts (0 shown as 0). Bottom: row-normalized probability (0 -> NaN).
    im_c = plot_matrix_nan(axes[0, k], c, BOUT_CALL_TYPES,
                           f"{loc} - counts ({GAP_LO_S}-{GAP_HI_S}s, n={int(c.values.sum()):,})",
                           vmax=count_vmax, cmap_name="Blues",
                           is_count=True, zero_as_nan=False)
    im_r = plot_matrix_nan(axes[1, k], r, BOUT_CALL_TYPES,
                           f"{loc} - P(next | current), row-normalized",
                           vmax=1.0, cmap_name="Greens", zero_as_nan=True)
fig.colorbar(im_c, ax=axes[0, :], fraction=0.025, pad=0.02).set_label("transition count")
fig.colorbar(im_r, ax=axes[1, :], fraction=0.025, pad=0.02).set_label("P(next | current)")
fig.suptitle(f"Bout -> bout transitions by location ({GAP_LO_S}-{GAP_HI_S} s)", y=0.99, fontsize=13)
plt.show()

## Conversation structure — enrichment over chance (log2 O/E)

Row-normalized P(next|current) is dominated by **base rates**: every row leans toward
whichever type is common (stacks), so it mostly shows marginal frequency, not sequencing.
To see *which bout actually follows which beyond chance*, compare observed transition
counts to those expected if current and next were independent:

`expected[a→b] = rowsum(a) · colsum(b) / total`,  shown as **log2(observed / expected)**.

> **0** = exactly as often as chance · **>0 (red)** = b follows a *more* than chance ·
> **<0 (blue)** = *less* than chance. Cells with zero observed transitions are NaN.

The colour scale is set from the **off-diagonal** cells; the diagonal (self-perseveration)
is usually strongly positive and may clip — its true value is printed in the cell.

In [ ]:
def enrichment_log2(counts):
    """log2(observed / expected) under independence; NaN where observed or expected is 0."""
    C = counts.to_numpy(dtype=float)
    total = C.sum()
    expected = np.outer(C.sum(axis=1), C.sum(axis=0)) / total
    with np.errstate(divide="ignore", invalid="ignore"):
        L = np.log2(C / expected)
    L[(C == 0) | (expected == 0)] = np.nan
    return pd.DataFrame(L, index=counts.index, columns=counts.columns)


def plot_enrichment(ax, M, order, title, vmax):
    A = M.reindex(index=order, columns=order).to_numpy(dtype=float)
    cmap = plt.get_cmap("RdBu_r").copy(); cmap.set_bad("#e8e8e8")
    im = ax.imshow(A, cmap=cmap, vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=40, ha="right")
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
    for i in range(len(order)):
        for j in range(len(order)):
            v = A[i, j]
            ax.text(j, i, "NaN" if np.isnan(v) else f"{v:+.2f}", ha="center", va="center",
                    fontsize=8, color="#999" if np.isnan(v) else "black")
    ax.set_title(title, fontsize=10); ax.set_xlabel("next bout"); ax.set_ylabel("current bout")
    return im


# Reuse pooled `counts` (Step 2) and per-location `loc_counts` (cell above).
enr = {"pooled (all locations)": enrichment_log2(counts),
       "arena":                  enrichment_log2(loc_counts["arena"]),
       "underground":            enrichment_log2(loc_counts["underground"])}

# Colour scale from OFF-diagonal magnitudes (diagonal self-perseveration can be extreme).
offdiag = []
for E in enr.values():
    V = E.to_numpy(dtype=float).copy(); np.fill_diagonal(V, np.nan)
    offdiag.append(np.nanmax(np.abs(V)))
vmax = float(np.nanmax(offdiag))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (name, E) in zip(axes, enr.items()):
    im = plot_enrichment(ax, E, BOUT_CALL_TYPES, name, vmax)
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02).set_label("log2(observed / expected)")
fig.suptitle(f"Bout -> bout conversation structure: enrichment over chance "
             f"({GAP_LO_S}-{GAP_HI_S} s)\n"
             f"red = follows MORE than chance, blue = LESS; diagonal may clip (value printed)",
             y=1.05, fontsize=12)
plt.show()

## Tau-resolved transitions — P(next type | current type) vs the gap tau

The fixed-band matrix collapses every gap in (2, 300] s into one number. Here we let the
gap **tau** vary: for each *current* type (one panel) we plot, against tau (log axis),
`P(next type = Y | current type, gap ≈ tau)` — one line per *next* type Y. Lines within a
panel sum to 1 at each tau (bins with too few transitions are left blank).

Computed at **both units**:
- **Call-level** — events are individual calls; generalises the call-transition
  notebook's three fixed bands (0.05 / 2 / 300 s, marked) into continuous curves.
- **Bout-level** — events are bouts; the tau-resolved version of the bout matrix.

The (2, 300] s band is shaded for reference.

In [ ]:
TAU_BINS    = np.logspace(np.log10(0.03), np.log10(600), 28)   # log-spaced gap bins (s)
TAU_CENTERS = np.sqrt(TAU_BINS[:-1] * TAU_BINS[1:])           # geometric bin centres
TAU_MARKERS = [0.05, GAP_LO_S, GAP_HI_S]                      # band edges to mark


def transition_pairs(events, type_order, group_cols, start_col, stop_col,
                     type_col="event_type"):
    """All consecutive transitions among `type_order` events, with their gap.

    Events are filtered to `type_order`, ordered by start within each group, and each
    adjacent pair yields (curr_type, next_type, tau_s = next.start - curr.stop).
    Adjacency is computed AFTER filtering, so "next" means the next event of these types.
    """
    ev = events[events[type_col].isin(type_order)]
    parts = []
    for _, g in ev.groupby(list(group_cols)):
        g = g.sort_values(start_col)
        t  = g[type_col].to_numpy()
        st = g[start_col].to_numpy()
        sp = g[stop_col].to_numpy()
        tau = (st[1:] - sp[:-1]) / np.timedelta64(1, "s")
        parts.append(pd.DataFrame({"curr_type": t[:-1], "next_type": t[1:], "tau_s": tau}))
    return pd.concat(parts, ignore_index=True)


def plot_tau_curves(pairs, type_order, title, tau_bins=TAU_BINS, min_count=30):
    """One panel per current type; one line per next type; x = gap tau (log)."""
    pairs = pairs[pairs["tau_s"] > 0].copy()
    pairs["bin"] = np.digitize(pairs["tau_s"].to_numpy(), tau_bins) - 1
    pairs = pairs[(pairs["bin"] >= 0) & (pairs["bin"] < len(tau_bins) - 1)]
    centers = np.sqrt(tau_bins[:-1] * tau_bins[1:])

    fig, axes = plt.subplots(1, len(type_order),
                             figsize=(3.7 * len(type_order), 3.6), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, X in zip(axes, type_order):
        sub = pairs[pairs["curr_type"] == X]
        counts = np.zeros((len(centers), len(type_order)))
        for j, Y in enumerate(type_order):
            s = sub[sub["next_type"] == Y].groupby("bin").size()
            counts[s.index.to_numpy(), j] = s.to_numpy()
        total = counts.sum(axis=1)
        with np.errstate(invalid="ignore", divide="ignore"):
            prob = counts / total[:, None]
        prob[total < min_count] = np.nan        # blank out under-sampled bins
        for j, Y in enumerate(type_order):
            ax.plot(centers, prob[:, j], marker="o", ms=3, lw=1.5,
                    color=TYPE_COLORS.get(Y), label=Y)
        ax.axvspan(GAP_LO_S, GAP_HI_S, color="gray", alpha=0.10)
        for mx in TAU_MARKERS:
            ax.axvline(mx, color="gray", ls=":", lw=0.8)
        ax.set_xscale("log"); ax.set_ylim(0, 1)
        ax.set_title(f"current = {X}  (n={int(total.sum()):,})", fontsize=10)
        ax.set_xlabel("gap tau (s)")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0].set_ylabel("P(next type | current, tau)")
    axes[-1].legend(title="next type", fontsize=8, loc="upper right")
    fig.suptitle(title, y=1.03, fontsize=12)
    fig.tight_layout()
    return fig

### By location — arena vs underground

The same tau curves, split into **arena** (top row) and **underground** (bottom row).
Columns are the *current* type; lines are the *next* type; rows share axes. Computed for
both units (call→call and bout→bout).

In [ ]:
def loc_split_pairs(events, type_order, group_cols, start_col, stop_col, loc_groups):
    """Build transition pairs separately for arena and underground."""
    ev = events.assign(_loc2=events["assigned_location"].map(loc_groups))
    return {loc: transition_pairs(ev[ev["_loc2"] == loc], type_order, group_cols,
                                  start_col, stop_col)
            for loc in ["arena", "underground"]}


def plot_tau_curves_by_loc(pairs_by_loc, type_order, title, tau_bins=TAU_BINS, min_count=30):
    """Grid of tau curves: rows = location, cols = current type, lines = next type."""
    rows = list(pairs_by_loc)
    centers = np.sqrt(tau_bins[:-1] * tau_bins[1:])
    fig, axes = plt.subplots(len(rows), len(type_order),
                             figsize=(3.3 * len(type_order), 3.0 * len(rows)),
                             sharex=True, sharey=True, squeeze=False)
    for r, loc in enumerate(rows):
        P = pairs_by_loc[loc]
        P = P[P["tau_s"] > 0].copy()
        P["bin"] = np.digitize(P["tau_s"].to_numpy(), tau_bins) - 1
        P = P[(P["bin"] >= 0) & (P["bin"] < len(tau_bins) - 1)]
        for c, X in enumerate(type_order):
            ax = axes[r, c]
            sub = P[P["curr_type"] == X]
            counts = np.zeros((len(centers), len(type_order)))
            for j, Y in enumerate(type_order):
                s = sub[sub["next_type"] == Y].groupby("bin").size()
                counts[s.index.to_numpy(), j] = s.to_numpy()
            total = counts.sum(axis=1)
            with np.errstate(invalid="ignore", divide="ignore"):
                prob = counts / total[:, None]
            prob[total < min_count] = np.nan
            for j, Y in enumerate(type_order):
                ax.plot(centers, prob[:, j], marker="o", ms=2.5, lw=1.3,
                        color=TYPE_COLORS.get(Y), label=Y)
            ax.axvspan(GAP_LO_S, GAP_HI_S, color="gray", alpha=0.10)
            for mx in TAU_MARKERS:
                ax.axvline(mx, color="gray", ls=":", lw=0.7)
            ax.set_xscale("log"); ax.set_ylim(0, 1)
            ax.text(0.96, 0.94, f"n={int(total.sum()):,}", transform=ax.transAxes,
                    ha="right", va="top", fontsize=7, color="gray")
            if r == 0:
                ax.set_title(f"current = {X}", fontsize=10)
            if c == 0:
                ax.set_ylabel(f"{loc}\nP(next | current, tau)", fontsize=9)
            if r == len(rows) - 1:
                ax.set_xlabel("gap tau (s)")
            ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0, -1].legend(title="next type", fontsize=7, loc="upper right")
    fig.suptitle(title, y=1.01, fontsize=13)
    fig.tight_layout()
    return fig

In [ ]:
call_pairs_loc = loc_split_pairs(calls, BOUT_CALL_TYPES, GROUP_COLS,
                                "start_time_real", "stop_time_real", LOC_GROUPS)
plot_tau_curves_by_loc(call_pairs_loc, BOUT_CALL_TYPES,
                       "Call -> call transition probability vs gap tau, by location")
plt.show()

In [ ]:
bout_pairs_loc = loc_split_pairs(bouts, BOUT_CALL_TYPES, GROUP_COLS,
                                "start_time", "stop_time", LOC_GROUPS)
plot_tau_curves_by_loc(bout_pairs_loc, BOUT_CALL_TYPES,
                       "Bout -> bout transition probability vs gap tau, by location")
plt.show()

## Call-level transition matrices (2–300 s) — for comparison with bouts

The same 2–300 s transition analysis but on **individual calls** instead of bouts: among
the 4 call types, consecutive calls within each `(date, exp, location)` group whose gap is
in (2 s, 300 s]. Shown in the same formats as the bout matrices (counts + row-normalized
by location; and log2 O/E enrichment) so calls and bouts can be compared side by side.

In [ ]:
def counts_from_pairs(pairs, order, lo, hi):
    """Counts matrix of curr->next transitions with gap in (lo, hi]."""
    sub = pairs[(pairs["tau_s"] > lo) & (pairs["tau_s"] <= hi)]
    return (pd.crosstab(sub["curr_type"], sub["next_type"])
            .reindex(index=order, columns=order, fill_value=0).astype(int))


call_pairs = transition_pairs(calls, BOUT_CALL_TYPES, GROUP_COLS,
                              "start_time_real", "stop_time_real")          # all locations
call_counts = {
    "pooled (all locations)": counts_from_pairs(call_pairs, BOUT_CALL_TYPES, GAP_LO_S, GAP_HI_S),
    "arena":       counts_from_pairs(call_pairs_loc["arena"],       BOUT_CALL_TYPES, GAP_LO_S, GAP_HI_S),
    "underground": counts_from_pairs(call_pairs_loc["underground"], BOUT_CALL_TYPES, GAP_LO_S, GAP_HI_S),
}
print("call->call transitions (2-300s):", {k: int(v.values.sum()) for k, v in call_counts.items()})

In [ ]:
cvmax = max(call_counts["arena"].to_numpy().max(),
            call_counts["underground"].to_numpy().max())
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for k, loc in enumerate(["arena", "underground"]):
    c = call_counts[loc]
    r = c.div(c.sum(axis=1), axis=0)
    im_c = plot_matrix_nan(axes[0, k], c, BOUT_CALL_TYPES,
                           f"{loc} - call counts ({GAP_LO_S}-{GAP_HI_S}s, n={int(c.values.sum()):,})",
                           vmax=cvmax, cmap_name="Blues", is_count=True, zero_as_nan=False)
    im_r = plot_matrix_nan(axes[1, k], r, BOUT_CALL_TYPES,
                           f"{loc} - P(next call | current call), row-normalized",
                           vmax=1.0, cmap_name="Greens", zero_as_nan=True)
fig.colorbar(im_c, ax=axes[0, :], fraction=0.025, pad=0.02).set_label("transition count")
fig.colorbar(im_r, ax=axes[1, :], fraction=0.025, pad=0.02).set_label("P(next | current)")
fig.suptitle(f"CALL -> call transitions by location ({GAP_LO_S}-{GAP_HI_S} s)", y=0.99, fontsize=13)
plt.show()

In [ ]:
enr_call = {k: enrichment_log2(v) for k, v in call_counts.items()}
offdiag = []
for E in enr_call.values():
    V = E.to_numpy(dtype=float).copy(); np.fill_diagonal(V, np.nan)
    offdiag.append(np.nanmax(np.abs(V)))
vmax = float(np.nanmax(offdiag))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (name, E) in zip(axes, enr_call.items()):
    im = plot_enrichment(ax, E, BOUT_CALL_TYPES, name, vmax)
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02).set_label("log2(observed / expected)")
fig.suptitle(f"CALL -> call conversation structure: enrichment over chance "
             f"({GAP_LO_S}-{GAP_HI_S} s)\n"
             f"red = follows MORE than chance, blue = LESS; diagonal may clip",
             y=1.05, fontsize=12)
plt.show()

### Call-level transition matrices (5–20 s band)

The same call→call analysis restricted to a tighter gap window, **(5 s, 20 s]**, for
comparison with the 2–300 s version. Reuses `call_pairs` / `call_pairs_loc` from above;
only the band changes.

In [ ]:
CALL_GAP_LO_S, CALL_GAP_HI_S = 5, 20   # tighter inter-call gap window

call_counts_520 = {
    "pooled (all locations)": counts_from_pairs(call_pairs,                BOUT_CALL_TYPES, CALL_GAP_LO_S, CALL_GAP_HI_S),
    "arena":                  counts_from_pairs(call_pairs_loc["arena"],       BOUT_CALL_TYPES, CALL_GAP_LO_S, CALL_GAP_HI_S),
    "underground":            counts_from_pairs(call_pairs_loc["underground"], BOUT_CALL_TYPES, CALL_GAP_LO_S, CALL_GAP_HI_S),
}
print("call->call transitions (5-20s):", {k: int(v.values.sum()) for k, v in call_counts_520.items()})

In [ ]:
cvmax = max(call_counts_520["arena"].to_numpy().max(),
            call_counts_520["underground"].to_numpy().max())
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for k, loc in enumerate(["arena", "underground"]):
    c = call_counts_520[loc]
    r = c.div(c.sum(axis=1), axis=0)
    im_c = plot_matrix_nan(axes[0, k], c, BOUT_CALL_TYPES,
                           f"{loc} - call counts ({CALL_GAP_LO_S}-{CALL_GAP_HI_S}s, n={int(c.values.sum()):,})",
                           vmax=cvmax, cmap_name="Blues", is_count=True, zero_as_nan=False)
    im_r = plot_matrix_nan(axes[1, k], r, BOUT_CALL_TYPES,
                           f"{loc} - P(next call | current call), row-normalized",
                           vmax=1.0, cmap_name="Greens", zero_as_nan=True)
fig.colorbar(im_c, ax=axes[0, :], fraction=0.025, pad=0.02).set_label("transition count")
fig.colorbar(im_r, ax=axes[1, :], fraction=0.025, pad=0.02).set_label("P(next | current)")
fig.suptitle(f"CALL -> call transitions by location ({CALL_GAP_LO_S}-{CALL_GAP_HI_S} s)", y=0.99, fontsize=13)
plt.show()

In [ ]:
enr_call_520 = {k: enrichment_log2(v) for k, v in call_counts_520.items()}
offdiag = []
for E in enr_call_520.values():
    V = E.to_numpy(dtype=float).copy(); np.fill_diagonal(V, np.nan)
    offdiag.append(np.nanmax(np.abs(V)))
vmax = float(np.nanmax(offdiag))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (name, E) in zip(axes, enr_call_520.items()):
    im = plot_enrichment(ax, E, BOUT_CALL_TYPES, name, vmax)
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02).set_label("log2(observed / expected)")
fig.suptitle(f"CALL -> call conversation structure: enrichment over chance "
             f"({CALL_GAP_LO_S}-{CALL_GAP_HI_S} s)\n"
             f"red = follows MORE than chance, blue = LESS; diagonal may clip",
             y=1.05, fontsize=12)
plt.show()

## QC — eyeball example transitions as spectrograms

At this timescale bouts are far apart (median inter-bout gap ≈ 50 s), so we can't show one
continuous "2 s before/after the switch" window. Instead each example shows **two 2 s
clips**: the **end of bout A** and the **start of bout B**, each with the boundary (the
switch) marked at *t = 0* (cyan line). The actual gap is printed per example.

In [ ]:
QC_N_EXAMPLES   = 100
QC_PER_FIGURE   = 50
QC_SEED         = 0
QC_LOC          = None     # None = all; or "arena" / "underground"
QC_CROSS_ONLY   = False    # True = only transitions where the bout type changes
PRE_S = POST_S  = 2.0      # seconds shown before/after each boundary
SPEC_FMAX_KHZ   = 60       # display ceiling (audio Nyquist is 62.5 kHz)


def list_transitions(bouts, gap_lo_s, gap_hi_s, group_cols):
    """In-band transitions, carrying each side's audio coordinates for QC plotting."""
    out = []
    for _, g in bouts.groupby(list(group_cols)):
        g = g.sort_values("start_time").reset_index(drop=True)
        for i in range(len(g) - 1):
            a, b = g.iloc[i], g.iloc[i + 1]
            gap = (b["start_time"] - a["stop_time"]) / np.timedelta64(1, "s")
            if gap_lo_s < gap <= gap_hi_s:
                out.append(dict(
                    date_folder=a["date_folder"], exp=int(a["exp"]),
                    assigned_location=a["assigned_location"],
                    a_type=a["event_type"], b_type=b["event_type"], gap_s=gap,
                    a_file=int(a["last_file"]),  a_ch=int(a["last_ch"]),  a_fsec=float(a["last_fsec"]),
                    b_file=int(b["first_file"]), b_ch=int(b["first_ch"]), b_fsec=float(b["first_fsec"]),
                ))
    return pd.DataFrame(out)


transitions = list_transitions(bouts, GAP_LO_S, GAP_HI_S, GROUP_COLS)
pool = transitions
if QC_LOC is not None:
    pool = pool[pool["assigned_location"].map(LOC_GROUPS) == QC_LOC]
if QC_CROSS_ONLY:
    pool = pool[pool["a_type"] != pool["b_type"]]

examples = pool.sample(min(QC_N_EXAMPLES, len(pool)), random_state=QC_SEED).reset_index(drop=True)
print(f"{len(transitions):,} in-band transitions; showing {len(examples)} "
      f"(loc={QC_LOC}, cross_only={QC_CROSS_ONLY})")

In [ ]:
import soundfile as sf
import librosa


@functools.lru_cache(maxsize=64)
def _samplerate(path_str):
    return sf.info(path_str).samplerate


def load_clip(date_folder, exp, channel, file_num, center_fsec):
    """(waveform, sr) for [center_fsec - PRE_S, center_fsec + POST_S] of a per-channel
    WAV, or (None, None) if the file is missing."""
    p = (BASE_AUDIO / date_folder / str(int(exp)) / "Averaged_wavs_w_annotations"
         / f"channel_{int(channel)}_file_{int(file_num):03d}.wav")
    if not p.exists():
        return None, None
    sr = _samplerate(str(p))
    start = max(0, int(round((center_fsec - PRE_S) * sr)))
    stop  = int(round((center_fsec + POST_S) * sr))
    y, _ = sf.read(str(p), start=start, stop=stop, dtype="float32", always_2d=False)
    return y, sr


def _spec_panel(ax, date_folder, exp, channel, file_num, center_fsec, label):
    y, sr = load_clip(date_folder, exp, channel, file_num, center_fsec)
    if y is None or len(y) < 512:
        ax.text(0.5, 0.5, "no audio", ha="center", va="center", fontsize=7)
        ax.set_xticks([]); ax.set_yticks([]); return
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=512, hop_length=128)), ref=np.max)
    ax.imshow(S, origin="lower", aspect="auto",
              extent=[-PRE_S, POST_S, 0, sr / 2 / 1000], cmap="magma", vmin=-80, vmax=0)
    ax.axvline(0, color="cyan", lw=1.0)                  # the switch (boundary)
    ax.set_ylim(0, SPEC_FMAX_KHZ)
    ax.text(0.03, 0.92, label, transform=ax.transAxes, color="white", fontsize=7,
            va="top", ha="left")
    ax.tick_params(labelsize=6)


def plot_transition_examples(examples, cols_examples=5):
    """Grid of bout transitions: per example two panels (A end | B start)."""
    n = len(examples)
    rows = int(np.ceil(n / cols_examples))
    fig, axes = plt.subplots(rows, cols_examples * 2,
                             figsize=(cols_examples * 2 * 1.9, rows * 1.7), squeeze=False)
    for k in range(rows * cols_examples):
        r, c = divmod(k, cols_examples)
        ax_a, ax_b = axes[r, 2 * c], axes[r, 2 * c + 1]
        if k >= n:
            ax_a.axis("off"); ax_b.axis("off"); continue
        e = examples.iloc[k]
        _spec_panel(ax_a, e.date_folder, e.exp, e.a_ch, e.a_file, e.a_fsec, f"{e.a_type} end")
        _spec_panel(ax_b, e.date_folder, e.exp, e.b_ch, e.b_file, e.b_fsec, f"{e.b_type} start")
        loc = LOC_GROUPS.get(e.assigned_location, e.assigned_location)
        ax_a.set_ylabel(f"{e.a_type}->{e.b_type}\n{e.gap_s:.0f}s · {loc}", fontsize=6)
    fig.suptitle(f"Bout transitions (cyan = switch, +/-{PRE_S:g}s)", y=1.002, fontsize=12)
    fig.tight_layout()
    return fig


if BASE_AUDIO is None:
    print("Raw WAVs not available on this platform - run the QC figures on the cluster.")

In [ ]:
# Figure 1: first QC_PER_FIGURE examples.
if BASE_AUDIO is not None:
    plot_transition_examples(examples.iloc[:QC_PER_FIGURE])
    plt.show()

In [ ]:
# Figure 2: next QC_PER_FIGURE examples.
if BASE_AUDIO is not None and len(examples) > QC_PER_FIGURE:
    plot_transition_examples(examples.iloc[QC_PER_FIGURE:2 * QC_PER_FIGURE])
    plt.show()

## Sanity check — inter-bout gap distribution

Where does the 2–300 s band sit among all inter-bout gaps? Per the project convention for
heavy-tailed data, gaps are log10-transformed and binned linearly with `density=True`.

In [ ]:
all_gaps = []
for _, g in bouts.groupby(GROUP_COLS):
    g = g.sort_values("start_time")
    gaps = (g["start_time"].to_numpy()[1:] - g["stop_time"].to_numpy()[:-1]) / np.timedelta64(1, "s")
    all_gaps.append(gaps[gaps > 0])
all_gaps = np.concatenate(all_gaps)

fig, ax = plt.subplots(figsize=(10, 4))
log_gaps = np.log10(all_gaps)
ax.hist(log_gaps, bins=np.linspace(log_gaps.min(), log_gaps.max(), 80),
        density=True, color="#6C757D", edgecolor="white", linewidth=0.3)
ax.axvspan(np.log10(GAP_LO_S), np.log10(GAP_HI_S), color="#457B9D", alpha=0.12)
for thr in (GAP_LO_S, GAP_HI_S):
    ax.axvline(np.log10(thr), color="#E63946", linestyle="--", linewidth=1.3)
    frac = (all_gaps <= thr).mean()
    ax.text(np.log10(thr), ax.get_ylim()[1] * 0.95, f" {thr}s ({frac*100:.0f}% <=)",
            color="#E63946", rotation=90, va="top", ha="left", fontsize=9)
ticks = np.array([0.1, 1, 10, 60, 300, 3600])
ax.set_xticks(np.log10(ticks)); ax.set_xticklabels([f"{t:g}" for t in ticks])
ax.set_xlabel("Inter-bout silent gap (sec, log scale)"); ax.set_ylabel("Density")
ax.set_title(f"Inter-bout gaps (n={len(all_gaps):,})  -  {GAP_LO_S}-{GAP_HI_S}s band shaded")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()